# Grazioso Salvare Rescue Dog Dashboard
CS-340 Project Two | Ziyan Noorani

MVC layout:
* **Model:** MongoDB `aac.animals`
* **View:** Dash widgets (filter, data table, map, pie chart)
* **Controller:** Dash callbacks that call the `AnimalShelter` CRUD module

In [1]:
# ---------------------------------------------------------------- Imports
import base64

import dash_leaflet as dl
import pandas as pd
import plotly.express as px
from dash import Dash, dash_table, dcc, html
from dash.dependencies import Input, Output

# CRUD module from Project One (CRUD_Python_Module.py, same folder)
from CRUD_Python_Module import AnimalShelter

In [2]:
from getpass import getpass
# ---------------------------------------------------------------- Model
# Hard-coded aacuser credentials, as the project directions require
USERNAME = "aacuser"
PASSWORD = getpass("Password for aacuser: ")  # kept out of the code since this notebook is public on GitHub

# CRUD object named db to match the SNHU starter code
db = AnimalShelter(USERNAME, PASSWORD)

# "Retrieve all" query feeds the starting (unfiltered) state
df = pd.DataFrame.from_records(db.read({}))

# MongoDB's _id is an ObjectId, which the Dash DataTable cannot display
df = df.drop(columns=["_id"], errors="ignore")
print(f"Loaded {len(df)} records")

Loaded 10000 records


In [3]:
# ---------------------------------------------------------------- Filter queries
# Each rescue type maps to one MongoDB query built from the
# Rescue Type and Preferred Dog Breeds table in the spec.
# Breeds use an anchored regex so mixes such as "German Shepherd Mix"
# or "Newfoundland/Labrador Retriever" still match.
# The shelter data truncates Doberman Pinscher to "Doberman Pinsch".

def build_query(breeds, sex, min_weeks, max_weeks):
    """Return a MongoDB query for dogs of the given breeds, sex, and age range."""
    return {
        "animal_type": "Dog",
        "breed": {"$regex": "^(" + "|".join(breeds) + ")"},
        "sex_upon_outcome": sex,
        "age_upon_outcome_in_weeks": {"$gte": min_weeks, "$lte": max_weeks},
    }


RESCUE_QUERIES = {
    "water": build_query(
        ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"],
        "Intact Female", 26, 156),
    "mountain": build_query(
        ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
         "Siberian Husky", "Rottweiler"],
        "Intact Male", 26, 156),
    "disaster": build_query(
        ["Doberman Pinsch", "German Shepherd", "Golden Retriever",
         "Bloodhound", "Rottweiler"],
        "Intact Male", 20, 300),
    "reset": {},  # empty query = every record
}


def get_filtered_df(filter_type):
    """Run the matching query through the CRUD module and return a DataFrame."""
    query = RESCUE_QUERIES.get(filter_type, {})
    records = db.read(query)
    dff = pd.DataFrame.from_records(records)
    # Drop the ObjectId column; errors="ignore" covers an empty result
    return dff.drop(columns=["_id"], errors="ignore")

In [4]:
# ---------------------------------------------------------------- View
app = Dash(__name__)

# Logo is embedded as base64 so the page needs no static file server
LOGO_FILE = "Grazioso Salvare Logo.png"
with open(LOGO_FILE, "rb") as logo:
    encoded_logo = base64.b64encode(logo.read()).decode()

app.layout = html.Div([
    # Branding: logo links to the client home page, plus unique identifier
    html.Div([
        html.A(
            html.Img(src=f"data:image/png;base64,{encoded_logo}",
                     style={"height": "110px"}),
            href="https://www.snhu.edu", target="_blank"),
        html.Div([
            html.H1("Grazioso Salvare Rescue Dog Dashboard",
                    style={"margin": "0"}),
            html.H4("Created by Ziyan Noorani | CS-340",
                    id="unique-identifier", style={"margin": "4px 0 0 0"}),
        ], style={"marginLeft": "20px"}),
    ], style={"display": "flex", "alignItems": "center"}),
    html.Hr(),

    # Interactive filter options
    html.Div([
        html.Label("Filter by rescue type:", style={"fontWeight": "bold"}),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"},
                {"label": "Reset", "value": "reset"},
            ],
            value="reset",
            inline=True,
            labelStyle={"marginRight": "20px"},
        ),
        html.Div(id="record-count", style={"marginTop": "6px"}),
    ]),
    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": c, "id": c, "deletable": False, "selectable": True}
                 for c in df.columns],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "minWidth": "90px"},
        style_header={"backgroundColor": "#c8102e", "color": "white",
                      "fontWeight": "bold"},
    ),
    html.Br(),
    html.Hr(),

    # Charts side by side
    html.Div([
        html.Div(id="graph-id", style={"flex": "1"}),
        html.Div(id="map-id", style={"flex": "1"}),
    ], style={"display": "flex", "gap": "20px"}),
])

In [5]:
# ---------------------------------------------------------------- Controller
@app.callback(
    [Output("datatable-id", "data"),
     Output("datatable-id", "selected_rows"),
     Output("record-count", "children")],
    [Input("filter-type", "value")])
def update_dashboard(filter_type):
    """Re-query MongoDB when the user picks a rescue type."""
    dff = get_filtered_df(filter_type)
    count = f"{len(dff)} matching records"
    # Reset the selected row so the map always points at a valid record
    return dff.to_dict("records"), [0] if len(dff) else [], count


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")])
def update_graphs(view_data):
    """Pie chart of breeds in the rows currently shown in the table."""
    if not view_data:
        return html.P("No records to chart.")
    dff = pd.DataFrame.from_dict(view_data)
    # Keep the pie readable: top 10 breeds, everything else as Other
    counts = dff["breed"].value_counts()
    top = counts.head(10)
    if len(counts) > 10:
        top["Other"] = counts.iloc[10:].sum()
    fig = px.pie(names=top.index, values=top.values,
                 title="Breed Distribution",
                 color_discrete_sequence=px.colors.sequential.RdBu)
    return [dcc.Graph(figure=fig)]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")])
def update_styles(selected_columns):
    """Shade any column the user selects."""
    if not selected_columns:
        return []
    return [{"if": {"column_id": col}, "background_color": "#f6d6db"}
            for col in selected_columns]


@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")])
def update_map(view_data, index):
    """Geolocation chart: marker for the selected row in the current view."""
    if not view_data:
        return html.P("No location to map.")
    dff = pd.DataFrame.from_dict(view_data)
    row = index[0] if index else 0
    if row >= len(dff):
        row = 0
    lat = dff.iloc[row]["location_lat"]
    lon = dff.iloc[row]["location_long"]
    name = dff.iloc[row]["name"]
    # mongoimport stores a blank name as "", pandas may show NaN
    if pd.isna(name) or str(name).strip() == "":
        name = "No name"
    return [
        dl.Map(style={"width": "100%", "height": "450px"},
               center=[lat, lon], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(position=[lat, lon], children=[
                       dl.Tooltip(dff.iloc[row]["breed"]),
                       dl.Popup([html.H3("Animal Name"), html.P(name)]),
                   ]),
               ])
    ]

In [ ]:
# ---------------------------------------------------------------- Run
# Codio: open https://<your-box>-8050.codio.io once the server starts
# Dash 3 removed run_server; older Codio Dash versions only have run_server
run = getattr(app, "run", None) or app.run_server
run(host="0.0.0.0", port=8050, debug=False)

Dash is running on http://0.0.0.0:8050/

 * Serving Flask app '__main__' (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on all addresses.
 * Running on http://10.17.171.213:8050/ (Press CTRL+C to quit)
192.168.11.136 - - [14/Oct/2026 22:10:40] "GET / HTTP/1.1" 200 -
192.168.11.136 - - [14/Oct/2026 22:10:40] "GET /_dash-component-suites/dash/deps/polyfill@7.v2_8_1m1752168217.12.1.min.js HTTP/1.1" 200 -
192.168.11.209 - - [14/Oct/2026 22:10:40] "GET /_dash-component-suites/dash/deps/react-dom@16.v2_8_1m1752168217.14.0.min.js HTTP/1.1" 200 -
192.168.11.209 - - [14/Oct/2026 22:10:40] "GET /_dash-component-suites/dash/deps/prop-types@15.v2_8_1m1752168217.8.1.min.js HTTP/1.1" 200 -
192.168.11.136 - - [14/Oct/2026 22:10:40] "GET /_dash-component-suites/dash/html/dash_html_components.v2_0_8m1752168217.min.js HTTP/1.1" 200 -
192.168.10.208 - - [14/Oct/2026 22:10:40] "GET /_dash-component-suites/dash/dash_table/bundle.v5_2_2m1752168217.js HTTP/1.1" 200 -
192.168.11.209 - - [14/Oct/2026 22:10:40] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v2_8_1m1752168217.min.js HTTP/1.1" 200 -